# Tahap 1: Membangun Case Base
### CBR Putusan Wanprestasi — Direktori Putusan Mahkamah Agung RI

**Tim:**
- Ahmad Qayyim — 202310370311286
- Bintang Mars Satria Tuhu — 202310370311410

**Mata Kuliah:** Penalaran Komputer — SubCPMK-3: Case-Based Reasoning

---

Notebook ini menjalankan tahap 1 dari pipeline CBR:
1. Scraping 35 putusan wanprestasi dari Direktori MA RI
2. Ekstraksi PDF ke teks plain
3. Cleaning dan validasi teks
4. Penyimpanan ke Google Drive

## 1. Setup Environment

In [ ]:
!pip install requests beautifulsoup4 lxml pdfminer.six tqdm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/cbr-putusan-wanprestasi')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

for sub in ['src', 'data/raw/pdf', 'data/raw/html', 'data/raw/text',
            'data/cleaned', 'data/processed', 'data/eval', 'data/results',
            'logs', 'reports/figures']:
    (PROJECT_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'Working dir: {os.getcwd()}')

## 2. Upload Script Files

Pastikan file `utils.py`, `01_download_cases.py`, dan `02_extract_and_clean.py` sudah ada di folder `src/`. Bisa upload manual lewat panel Files Colab, atau git clone repository.

In [ ]:
# Cek file scripts sudah ada
required_files = ['src/utils.py', 'src/01_download_cases.py', 'src/02_extract_and_clean.py']
for f in required_files:
    path = PROJECT_DIR / f
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  [{status}] {f}')

## 3. Scraping Putusan dari Direktori MA RI

Target: **35 putusan wanprestasi**. Estimasi waktu: ~20-25 menit.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR / 'src'))

# Import dengan workaround karena nama file diawali angka
import importlib.util

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

scraper_mod = load_module('scraper_mod', PROJECT_DIR / 'src' / '01_download_cases.py')
CourtScraper = scraper_mod.CourtScraper

In [ ]:
scraper = CourtScraper(query='wanprestasi', target=35, delay=3.0)
results = scraper.run()

print(f'\nTotal di-scrape: {len(results)}')
ok_count = sum(1 for r in results if r.status == 'OK')
print(f'Berhasil OK   : {ok_count}')

## 4. Inspeksi Hasil Scraping

In [ ]:
import json
import pandas as pd

with open('data/raw/metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

df_meta = pd.DataFrame(metadata)
print(f'Shape: {df_meta.shape}')
print(f'\nDistribusi status:')
print(df_meta['status'].value_counts())
df_meta.head(10)

In [ ]:
# Lihat list PDF yang berhasil di-download
pdf_files = sorted((PROJECT_DIR / 'data' / 'raw' / 'pdf').glob('*.pdf'))
print(f'Total PDF: {len(pdf_files)}')
for p in pdf_files[:5]:
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name}  ({size_kb:.1f} KB)')

## 5. Ekstraksi PDF + Cleaning

Estimasi waktu: ~5-10 menit.

In [ ]:
cleaner_mod = load_module('cleaner_mod', PROJECT_DIR / 'src' / '02_extract_and_clean.py')
results_clean = cleaner_mod.process_all()

print(f'\nTotal diproses: {len(results_clean)}')

## 6. Analisis Kualitas Hasil Cleaning

In [ ]:
with open('data/raw/processing_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)

df_summary = pd.DataFrame(summary)
print('Distribusi extraction status:')
print(df_summary['extraction_status'].value_counts())
print('\nDistribusi validation status:')
print(df_summary['validation_status'].value_counts())
print(f'\nStatistik word count (setelah cleaning):')
print(df_summary['word_count_clean'].describe())

In [ ]:
# Sample preview teks bersih
sample = df_summary[df_summary['validation_status'] == 'VALID'].iloc[0]
if sample['cleaned_file']:
    with open(sample['cleaned_file'], 'r', encoding='utf-8') as f:
        text = f.read()
    print(f"Case: {sample['case_id']}")
    print(f"Word count: {sample['word_count_clean']}")
    print('--- PREVIEW (first 1500 chars) ---')
    print(text[:1500])

## 7. Visualisasi Statistik Case Base

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

valid_df = df_summary[df_summary['extraction_status'] == 'OK']
axes[0].hist(valid_df['word_count_clean'], bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(500, color='red', linestyle='--', label='Min 500 kata')
axes[0].set_title('Distribusi Word Count per Dokumen')
axes[0].set_xlabel('Jumlah kata')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

val_counts = df_summary['validation_status'].value_counts()
colors = ['#2ecc71' if s == 'VALID' else '#e74c3c' for s in val_counts.index]
axes[1].bar(val_counts.index, val_counts.values, color=colors, edgecolor='black')
axes[1].set_title('Status Validasi per Dokumen')
axes[1].set_ylabel('Jumlah')
for i, v in enumerate(val_counts.values):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('reports/figures/case_base_stats.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Final Checklist

Tahap 1 selesai. Pastikan:
- ≥ 30 dokumen ber-status `VALID`
- Folder `data/raw/pdf/` terisi PDF
- Folder `data/cleaned/` terisi .txt bersih
- File `data/raw/metadata.json` lengkap
- File `data/raw/processing_summary.json` lengkap
- Log scraping dan cleaning ada di `logs/`

In [ ]:
valid_count = (df_summary['validation_status'] == 'VALID').sum()
print(f'Final case base size: {valid_count} dokumen VALID')
if valid_count >= 30:
    print('STATUS: SIAP lanjut ke Tahap 2 (Case Representation)')
else:
    print('STATUS: KURANG. Perlu scrape ulang dengan target lebih besar.')